# Galactic-binary parameter inference with `jexplore`

Bayesian inference of the parameters of a **single galactic binary (GB)** from a
one-month LISA A/E/T frequency-domain datastream, using the affine-invariant
ensemble sampler [`jexplore`](https://pypi.org/project/jexplore/).

The datastream (clean signal + instrumental noise) and the noise PSD are produced
**entirely by the helpers in [`src/lisa.py`](src/lisa.py)** — `clean_signal`,
`sample_noise` and `noise_psd` — the same code that feeds the diffusion model in
[train.py](train.py). This notebook is the MCMC cross-check of that pipeline.

We sample the four parameters **`θ = [f₀, ḟ, A, ψ]`**, holding the sky position and
orientation `(ra, dec, ι, φ₀)` fixed at their injected values, and recover **all
four constrained**.

Inspired by [glitch_and_gb.ipynb](glitch_and_gb.ipynb) but with the glitch sector
removed — here there is *only* a galactic binary.

## Choosing a constrainable injection

`ḟ` is only measurable if the frequency chirp builds up enough phase over the
observation: its precision scales as `σ_ḟ ∝ 1 / (SNR · T_obs²)`. With the
`lisa.prior_inverse_cdf` training scale (`ḟ ≤ 4×10⁻¹⁸`) over a short baseline the
chirp is unresolvable and the `ḟ` marginal collapses onto the prior.

So, exactly as `glitch_and_gb.ipynb` does (it uses a *heavy-chirp* `ḟ = 10⁻¹⁷` over a
**1-year** baseline and **raises the `ḟ`/`A` prior caps** above the training prior),
we pick a heavy chirp here too. We run at **1-month** resolution (this GPU only has
~2 GB free; the year-long grid OOMs), and since `σ_ḟ ∝ 1/T_obs²` a month needs a
correspondingly heavier chirp `ḟ = 10⁻¹⁶` to reach the same resolvability — then all
four parameters are data-constrained. `f₀`, `A`, `ψ` are constrained at the
`lisa.prior_inverse_cdf` scale already.

## Likelihood normalisation

`lisa.sample_noise` colours the frequency-domain noise as
`n_f = √S(f)·(z_r + i z_i)/√2` with `z ~ N(0,1)`, so each rfft bin has
`E[|n_f|²] = S(f)`. The matching Gaussian log-likelihood is therefore

`log L(θ) = −Σ_f Σ_ch |d_f − h_f(θ)|² / S(f)`

with `S = lisa.noise_psd(channel)` — the very PSD used to generate the noise. (At
the truth this gives `log L ≈ −(#bins × #channels)`, the expected χ² with 2 dof
per complex bin, which we check below.)

In [ ]:
import sys, os, time
sys.path.insert(0, os.path.abspath(""))   # so `import src.lisa` works from the notebook
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")  # allocate on demand (shared GPU)

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import jax.random as jr
import numpy as np
import corner
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

from src import lisa

from jexplore.sampler import JaxSampler, Steps
from jexplore.sampling import EpochMH, SamplingMH
from jexplore.steps import Stretch
from jexplore.backends import DefaultBackend

print("JAX backend:", jax.default_backend(), "| devices:", jax.devices())

## 1. Observation, true parameters and datastream

Everything below comes from `src/lisa.py`. The clean A/E/T signal is
`lisa.clean_signal` and the instrumental noise realisation is `lisa.sample_noise`;
both return the rFFT on the same cropped frequency grid, so `data = signal + noise`.

In [ ]:
# ---- observation ----
T_OBS = lisa.MONTH_s            # observation time (matches train.py)
DT    = lisa.SAMPLING_STEP_s    # Nyquist step for the 3 mHz band (~167 s)
N_SLOW = 256                    # points for the slow TDI response (clean_signal `n`)
NCROP  = 32                     # frequency-grid crop (clean_signal/sample_noise `ncrop`)
SEED   = 0

# ---- true GB parameters [f0, fdot, A, ra, dec, psi, iota, phi0] ----
# Inferred (heavy-chirp injection so fdot is resolvable over one month):
F0_TRUE,   FDOT_TRUE = 2.0e-3, 1.0e-16      # Hz, Hz/s
A_TRUE,    PSI_TRUE  = 1.0e-22, float(jnp.pi / 4)
# Fixed (not inferred):
RA_TRUE, DEC_TRUE, IOTA_TRUE, PHI0_TRUE = 1.0, -0.5, 1.0, 0.0

gb_params_true = jnp.array([[F0_TRUE, FDOT_TRUE, A_TRUE,
                             RA_TRUE, DEC_TRUE, PSI_TRUE, IOTA_TRUE, PHI0_TRUE]])

# ---- datastream = clean signal + noise (both from lisa) ----
signal = lisa.clean_signal(gb_params_true, t_obs=T_OBS, dt=DT, n=N_SLOW, ncrop=NCROP)
key = jr.key(SEED)
key, key_noise = jr.split(key)
noise = lisa.sample_noise(key_noise, t_obs=T_OBS, dt=DT, ncrop=NCROP)
data  = signal + noise   # (F, 3) complex, channels A/E/T

# ---- matching frequency grid + PSD (same crop as the data) ----
n_samples = int(T_OBS / DT)
freq = jnp.fft.rfftfreq(n_samples, DT)
freq = freq[: (len(freq) // NCROP) * NCROP]
assert freq.shape[0] == data.shape[0]
f_safe = jnp.where(freq > 0, freq, 1.0)                       # avoid f=0 in the PSD
psd  = jnp.stack([lisa.noise_psd(c)(f_safe) for c in "AET"], axis=-1)   # (F, 3)
mask = (freq > 0)[:, None]                                    # drop the DC bin

snr = float(jnp.sqrt(jnp.sum(jnp.where(mask, jnp.abs(signal) ** 2 / psd, 0.0))))
print(f"data shape {data.shape},  {freq.shape[0]} freq bins")
print(f"injected GB optimal SNR = {snr:.1f}")

## 2. Parameterisation, prior and likelihood

We sample in `θ = [log f₀, log ḟ, log A, ψ]` — log for the three log-uniform GB
amplitudes/frequencies (their `lisa.prior_inverse_cdf` priors are log-uniform), and
linear for the polarisation angle `ψ`. The prior is flat in `θ`; the `ḟ` and `A`
upper caps are raised above the `lisa.prior_inverse_cdf` training bounds to bracket
the heavy-chirp / louder injection (same trick as `glitch_and_gb.ipynb`).

In [ ]:
# Prior bounds — f0/psi as in lisa.prior_inverse_cdf; fdot/A caps raised to
# bracket the heavy-chirp / louder injection (cf. glitch_and_gb.ipynb).
F0_MIN,   F0_MAX   = 1e-4,  3e-3
FDOT_MIN, FDOT_MAX = 1e-22, 1e-15
A_MIN,    A_MAX    = 1e-25, 1.7e-22
PSI_MIN,  PSI_MAX  = 0.0,   float(jnp.pi)

DIM    = 4
labels = ["log f0", "log fdot", "log A", "psi"]

theta_true = jnp.array([jnp.log(F0_TRUE), jnp.log(FDOT_TRUE), jnp.log(A_TRUE), PSI_TRUE])


def to_params(theta):
    """Sampling vector θ -> full (1, 8) GB parameter array with fixed sky/orientation."""
    f0, fdot, A = jnp.exp(theta[0]), jnp.exp(theta[1]), jnp.exp(theta[2])
    psi = theta[3]
    return jnp.array([[f0, fdot, A, RA_TRUE, DEC_TRUE, psi, IOTA_TRUE, PHI0_TRUE]])


@jax.jit
def log_lik(theta):
    h = lisa.clean_signal(to_params(theta), t_obs=T_OBS, dt=DT, n=N_SLOW, ncrop=NCROP)
    r = data - h
    return -jnp.sum(jnp.where(mask, jnp.abs(r) ** 2 / psd, 0.0))


@jax.jit
def log_prior(theta):
    ok = (
        (theta[0] >= jnp.log(F0_MIN))   & (theta[0] <= jnp.log(F0_MAX))
      & (theta[1] >= jnp.log(FDOT_MIN)) & (theta[1] <= jnp.log(FDOT_MAX))
      & (theta[2] >= jnp.log(A_MIN))    & (theta[2] <= jnp.log(A_MAX))
      & (theta[3] >= PSI_MIN)           & (theta[3] <= PSI_MAX)
    )
    return jnp.where(ok, 0.0, -jnp.inf)


# Sanity: at the truth log L ≈ -(#bins × #channels) (χ², 2 dof per complex bin)
n_terms = int(jnp.sum(jnp.broadcast_to(mask, data.shape)))
print(f"log L(truth)      = {float(log_lik(theta_true)):.1f}")
print(f"-(bins×channels)  = {-n_terms}")
print(f"log L(truth+δ)    = {float(log_lik(theta_true + jnp.array([1e-3, 0., 0.1, 0.2]))):.1f}  (must be lower)")

## 3. Ensemble sampling with `jexplore`

A gradient-free affine-invariant `Stretch` ensemble. The `lisa.clean_signal` model
places the GB on an integer frequency bin (via `get_kmin`), so the likelihood is not
smoothly differentiable in `f₀` — a gradient-free ensemble move is the right choice.
Walkers are initialised in a tight ball around the (known) injection.

In [ ]:
N_WALKERS = 16
N_BURN    = 300
N_SAMP    = 1_000

# initialisation scatter per parameter (f0 is extremely well constrained → tiny)
sigma0 = jnp.array([1e-6, 0.1, 0.02, 0.02])
p0 = theta_true + sigma0 * jr.normal(jr.key(SEED + 1), (N_WALKERS, DIM))

sampling = SamplingMH(
    nwalker=N_WALKERS, temps=jnp.array([1.0]),
    loglik=log_lik, logprior=log_prior, dim=DIM,
)
steps   = Steps([{Stretch(permute=True).builder: 1.0}])
iepoch  = EpochMH({"p": p0})
backend = DefaultBackend(burn=N_BURN, inmem_epochs=1)

print(f"Running jexplore  ({N_WALKERS} walkers × {N_BURN + N_SAMP} iters)...")
t0 = time.time()
JaxSampler(sampling, steps, backend).run(iepoch, niters=N_BURN + N_SAMP, nepoch=1, seed=SEED + 1)
# backend stores (N_WALKERS, DIM, N_SAMP) -> flatten to (N_WALKERS*N_SAMP, DIM)
raw = backend.get_samples()["p"]
chain = np.asarray(raw.transpose(0, 2, 1).reshape(-1, DIM))
print(f"  done in {time.time() - t0:.1f} s,  {chain.shape[0]:,} samples")

In [ ]:
# Posterior summary in physical units
print(f"{'parameter':12s}  {'median':>14s}  {'truth':>14s}")
print("-" * 44)
meds = np.median(chain, axis=0)
phys = lambda v: (np.exp(v[0]), np.exp(v[1]), np.exp(v[2]), v[3])
names = ["f0 (Hz)", "fdot (Hz/s)", "A", "psi (rad)"]
for n, m, t in zip(names, phys(meds), phys(np.asarray(theta_true))):
    print(f"{n:12s}  {m:14.4e}  {t:14.4e}")

## 4. Corner plot

Marginal posteriors with `corner`, plotted in `[log₁₀ f₀, log₁₀ ḟ, log₁₀ A, ψ]`
(log₁₀ for the three log-uniform parameters). The injected truth is the black line.

All four marginals are localised on the injected truth (black). With the heavy-chirp
`ḟ = 10⁻¹⁶` the chirp is resolvable even over a month, so `ḟ` is now data-constrained
(log-`ḟ` posterior width ≈ 0.1 vs a prior width of ≈ 4.6) rather than prior-dominated.

In [ ]:
# Transform to plotting space: log10 for the log-uniform params, linear for psi
to_plot   = lambda a: np.column_stack([np.log10(np.exp(a[:, 0])),
                                       np.log10(np.exp(a[:, 1])),
                                       np.log10(np.exp(a[:, 2])),
                                       a[:, 3]])
plot_samples = to_plot(chain)
plot_truth   = [np.log10(F0_TRUE), np.log10(FDOT_TRUE), np.log10(A_TRUE), float(PSI_TRUE)]
plot_labels  = [r"$\log_{10} f_0$", r"$\log_{10}\dot f$", r"$\log_{10} A$", r"$\psi$"]

plt.close("all")
fig = corner.corner(
    plot_samples,
    labels=plot_labels,
    truths=plot_truth,
    truth_color="black",
    color="C0",
    show_titles=True,
    title_fmt=".3f",
    hist_kwargs={"density": True},
)
fig.legend(
    handles=[
        mlines.Line2D([], [], color="C0", label="jexplore posterior"),
        mlines.Line2D([], [], color="black", label="injected truth"),
    ],
    loc="upper right", fontsize=13, frameon=False,
)
fig.suptitle("Galactic-binary posterior (jexplore)", y=1.02)
#fig.savefig("GB_inference_corner.pdf", bbox_inches="tight")
plt.show()
#print("saved GB_inference_corner.pdf")

## 5. Flow-network posterior vs jexplore

Overlay of the **amortized flow-matching network** (trained by `train_Giorgio.py`)
against the `jexplore` MCMC posterior, on the *same* datastream `data` used above.

The network is conditioned on the WDM image of `data` (`gb_problem.datastream_to_y`)
— real inference of `p(params | data)` — and pushed from `Uniform[0,1]^4` to the
posterior with `gb_problem.sample_flow` (RK4, no `x % 1` wrap). It is only trained
for a few minutes, so it is **not** expected to reach the jexplore precision: the
point is to see it concentrating in the right region. jexplore (blue) is the tight
reference; the network (orange) is the broader learned posterior.

In [ ]:
import equinox as eqx
from src import gb_problem

# --- conditioning: WDM image of the SAME datastream jexplore used --------------
y_obs = gb_problem.datastream_to_y(data)

# --- load the trained network (must match train_Giorgio.py architecture) ------
key_net = jr.key(SEED + 7)
flow = gb_problem.build_flow(y_dim=y_obs.shape[-1], hidden_dim=256,
                             num_blocks=4, num_heads=8, key=key_net)
flow = eqx.tree_deserialise_leaves("checkpoint_Giorgio.eqx", flow)

# --- sample the network posterior --------------------------------------------
N_NET = chain.shape[0] if chain.shape[0] < 5000 else 5000

@eqx.filter_jit
def net_sample(x0, y):
    return jax.vmap(lambda xi: gb_problem.sample_flow(flow, xi, y))(x0)

x0 = jr.uniform(jr.key(SEED + 8), (N_NET, 1, gb_problem.X_DIM))
u_net = np.asarray(net_sample(x0, y_obs))[:, 0, :]               # (N, 4) unit cube
phys_net = np.asarray(gb_problem.u_to_physical(jnp.asarray(u_net)))   # (N, 4) physical
net_plot = np.column_stack([np.log10(phys_net[:, 0]), np.log10(phys_net[:, 1]),
                            np.log10(phys_net[:, 2]), phys_net[:, 3]])

# --- overlay corner: network (orange, broad) + jexplore (blue, tight) ---------
R = gb_problem.RANGES
plot_range = [(np.log10(R["f0"][0]),   np.log10(R["f0"][1])),
              (np.log10(R["fdot"][0]), np.log10(R["fdot"][1])),
              (np.log10(R["A"][0]),    np.log10(R["A"][1])),
              R["psi"]]

plt.close("all")
fig = corner.corner(net_plot, labels=plot_labels, truths=plot_truth,
                    truth_color="black", color="C1", range=plot_range,
                    hist_kwargs={"density": True})
corner.corner(plot_samples, fig=fig, color="C0", range=plot_range,
              hist_kwargs={"density": True})
fig.legend(handles=[
    mlines.Line2D([], [], color="C0", label="jexplore (MCMC)"),
    mlines.Line2D([], [], color="C1", label="flow network"),
    mlines.Line2D([], [], color="black", label="injected truth"),
], loc="upper right", fontsize=13, frameon=False)
fig.suptitle("GB posterior: flow network vs jexplore", y=1.02)
plt.show()